<sub>**The Annotated ML Handbook** · Part III: Pandas</sub>

# Chapter 11: pandas foundations: Series, DataFrames and selection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-3-pandas/11-pandas-basics.ipynb)

Companion code for the chapter. Every example from the book is here, in order, ready to run. The line-by-line explanations, definitions and discussion are in the book.

Run the setup cell first, then run the cells from top to bottom.

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

## 11.1 Series and DataFrame · *Beginner*

**Building both by hand**

In [2]:
import numpy as np
import pandas as pd

s = pd.Series([4.2, 3.9, 5.1], index=["a", "b", "c"], name="yield")
print(s)
print(s["b"], s.mean())

df = pd.DataFrame({
    "plot": ["P1", "P2", "P3", "P4"],
    "fertilizer": ["A", "B", "A", "B"],
    "rain_mm": [120, 95, 140, 80],
    "yield_t": [4.2, 3.9, 5.1, 3.4],
})
print(df)
print(df.shape, list(df.columns), df.index)

a    4.2
b    3.9
c    5.1
Name: yield, dtype: float64
3.9 4.3999999999999995
  plot fertilizer  rain_mm  yield_t
0   P1          A      120      4.2
1   P2          B       95      3.9
2   P3          A      140      5.1
3   P4          B       80      3.4
(4, 4) ['plot', 'fertilizer', 'rain_mm', 'yield_t'] RangeIndex(start=0, stop=4, step=1)


## 11.2 The practice dataset · *Beginner*

**Create and save the practice dataset**

In [3]:
rng = np.random.default_rng(2026)
n = 1000
age = rng.integers(18, 65, n)
sex = rng.choice(["female", "male"], n)
bmi = np.round(rng.normal(30, 6, n).clip(16, 53), 1)
children = rng.choice([0, 1, 2, 3, 4], n, p=[.43, .24, .18, .12, .03])
smoker = rng.choice(["yes", "no"], n, p=[.2, .8])
region = rng.choice(["northeast", "northwest", "southeast", "southwest"], n)
charges = (2500 + 260 * age + 320 * (bmi - 30).clip(0) + 450 * children
           + np.where(smoker == "yes", 23000, 0)
           + rng.gamma(2.0, 1500, n)).round(2)
ins = pd.DataFrame({"age": age, "sex": sex, "bmi": bmi, "children": children,
                    "smoker": smoker, "region": region, "charges": charges})
ins.loc[rng.choice(n, 40, replace=False), "bmi"] = np.nan
ins.to_csv("insurance.csv", index=False)
print("saved", ins.shape)

saved (1000, 7)


## 11.3 Reading and writing data · *Beginner*

**read_csv and friends**

In [4]:
df = pd.read_csv("insurance.csv")
print(df.head())

   age     sex   bmi  children smoker     region   charges
0   58  female  27.9         0     no  northwest  26115.49
1   26    male  39.4         2     no  northwest  15021.19
2   19    male  34.6         2     no  southwest  15319.15
3   48  female  17.3         2     no  southeast  16844.54
4   35    male  23.8         3     no  southwest  17184.32


## 11.4 Inspecting a DataFrame · *Beginner*

**First look**

In [5]:
print(df.shape)
df.info()

(1000, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1000 non-null   int64  
 1   sex       1000 non-null   str    
 2   bmi       960 non-null    float64
 3   children  1000 non-null   int64  
 4   smoker    1000 non-null   str    
 5   region    1000 non-null   str    
 6   charges   1000 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 54.8 KB


**Summary statistics**

In [6]:
print(df.describe().round(2))
print(df.describe(include="str"))

           age     bmi  children   charges
count  1000.00  960.00   1000.00   1000.00
mean     40.71   29.52      1.02  21473.52
std      13.50    5.84      1.16   9881.93
min      18.00   16.00      0.00   7375.92
25%      29.00   25.30      0.00  14669.86
50%      40.00   29.50      1.00  18670.80
75%      52.00   33.60      2.00  23315.51
max      64.00   50.00      4.00  50817.04
         sex smoker     region
count   1000   1000       1000
unique     2      2          4
top     male     no  southeast
freq     538    816        266


**Counts and missing values**

In [7]:
print(df["region"].value_counts())
print(df["smoker"].value_counts(normalize=True))
print(df.isna().sum())
print(df.nunique())

region
southeast    266
southwest    257
northeast    247
northwest    230
Name: count, dtype: int64
smoker
no     0.816
yes    0.184
Name: proportion, dtype: float64
age          0
sex          0
bmi         40
children     0
smoker       0
region       0
charges      0
dtype: int64
age           47
sex            2
bmi          244
children       5
smoker         2
region         4
charges     1000
dtype: int64


## 11.5 Selecting data · *Beginner*

### 11.5.1 Columns

**One column, several columns**

In [8]:
ages = df["age"]
print(type(ages).__name__, ages.shape)
sub = df[["age", "bmi", "charges"]]
print(type(sub).__name__, sub.shape)
print(df.age.mean())

Series (1000,)
DataFrame (1000, 3)
40.707


### 11.5.2 Rows: `.loc` (labels) and `.iloc` (positions)

**loc vs iloc**

In [9]:
print(df.loc[0, "charges"])
print(df.loc[0:2, ["age", "smoker"]])
print(df.iloc[0:2, 0:3])
print(df.iloc[-1])

26115.49
   age smoker
0   58     no
1   26     no
2   19     no
   age     sex   bmi
0   58  female  27.9
1   26    male  39.4
age                25
sex            female
bmi              27.6
children            0
smoker            yes
region      northwest
charges      33475.22
Name: 999, dtype: object


### 11.5.3 Filtering with conditions

**Boolean filtering**

In [10]:
smokers = df[df["smoker"] == "yes"]
print(len(smokers))
older_smokers = df[(df["age"] >= 50) & (df["smoker"] == "yes")]
print(len(older_smokers))
south = df[df["region"].isin(["southeast", "southwest"])]
print(south["region"].unique())
print(df.loc[df["bmi"] > 45, ["age", "bmi"]].head(3))
print(df.query("age < 25 and children >= 2").shape)
limit = 40
print(df.query("bmi > @limit").shape)

184
61
<StringArray>
['southwest', 'southeast']
Length: 2, dtype: str
     age   bmi
237   28  45.1
304   20  50.0
621   26  48.4
(40, 7)
(33, 7)


## 11.6 Adding, changing and removing columns · *Beginner*

**Creating new columns**

In [11]:
df = pd.read_csv("insurance.csv")
df["charges_k"] = df["charges"] / 1000
df["is_smoker"] = (df["smoker"] == "yes").astype(int)
df["age_group"] = pd.cut(df["age"], bins=[17, 30, 45, 64],
                         labels=["18-30", "31-45", "46-64"])
df = df.assign(log_charges=np.log(df["charges"]))
df.loc[df["children"] >= 3, "big_family"] = True
df = df.rename(columns={"bmi": "BMI"})
df = df.drop(columns=["big_family"])
print(df.head(3))

   age     sex   BMI  children smoker     region   charges  charges_k  \
0   58  female  27.9         0     no  northwest  26115.49   26.11549   
1   26    male  39.4         2     no  northwest  15021.19   15.02119   
2   19    male  34.6         2     no  southwest  15319.15   15.31915   

   is_smoker age_group  log_charges  
0          0     46-64    10.170284  
1          0     18-30     9.617217  
2          0     18-30     9.636859  


### 11.6.1 Sorting

**sort_values and nlargest**

In [12]:
print(df.sort_values("charges", ascending=False).head(3)[["age", "smoker", "charges"]])
print(df.sort_values(["smoker", "age"], ascending=[True, False]).head(3)[["smoker", "age"]])
print(df.nlargest(3, "BMI")[["BMI", "age"]])

     age smoker   charges
274   62    yes  50817.04
556   62    yes  50760.11
918   59    yes  48079.72
    smoker  age
50      no   64
131     no   64
160     no   64
      BMI  age
304  50.0   20
621  48.4   26
698  46.9   47


## 11.7 Copy-on-Write and the "chained assignment" trap · *Intermediate*

**This does NOT change df**

In [13]:
df = pd.read_csv("insurance.csv")
df[df["age"] > 60]["bmi"] = 0
print((df["bmi"] == 0).sum())

0


/tmp/ipykernel_17788/2665612345.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df[df["age"] > 60]["bmi"] = 0


**The correct way: one .loc call**

In [14]:
df.loc[df["age"] > 60, "bmi"] = 0
print((df["bmi"] == 0).sum())
df = pd.read_csv("insurance.csv")
young = df[df["age"] < 30].copy()
young["bmi"] = young["bmi"].round()

82


## 11.8 The index · *Intermediate*

**Setting and resetting the index**

In [15]:
idx = df.set_index("region")
print(idx.loc["southwest"].shape)
print(idx.reset_index().columns.tolist()[:3])
r = df[df["age"] > 60]
print(r.index[:5].tolist())
print(r.reset_index(drop=True).index[:5].tolist())

(257, 6)
['region', 'age', 'sex']
[21, 23, 50, 60, 62]
[0, 1, 2, 3, 4]


## 11.9 From pandas to NumPy and back · *Beginner*

**Getting arrays out for modelling**

In [16]:
X = df[["age", "bmi", "children"]].to_numpy()
y = df["charges"].to_numpy()
print(X.shape, X.dtype, y[:3])
back = pd.DataFrame(X[:2], columns=["age", "bmi", "children"])
print(back)

(1000, 3) float64 [26115.49 15021.19 15319.15]
    age   bmi  children
0  58.0  27.9       0.0
1  26.0  39.4       2.0


## Try it yourself

1. How many female smokers in the southeast are over 40?
2. Add a column `bmi_class` with "under" (< 18.5), "normal" (< 25), "over" (< 30), "obese" (≥ 30) using `pd.cut`.
3. Show the 5 youngest customers with 4 children, sorted by charges.

Worked solutions: [solutions-parts-1-4.ipynb](../../solutions/solutions-parts-1-4.ipynb), section *Chapter 11*.